<a href="https://colab.research.google.com/github/yi-juho/AI_LLM_BERT/blob/main/GLUE2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### model load/save

In [ ]:
%mkdir ./model

In [ ]:
path = '/content/model/'
torch.save(model.state_dict(), path+"RTE_128_3.pt")

In [ ]:
path = '/content/model/'
model.load_state_dict(torch.load(path + "QNLI_64.pt", map_location=device))
model.to(device)

### cordic new_64_128

In [ ]:
def float_to_fixed_point(value, int_bits=7, frac_bits=13):
    """
    실수를 32비트 Fixed-Point 바이너리로 변환하는 함수.
    - 31~20 비트: 항상 0 (상위 12비트)
    - 19~13 비트: 정수부 (7비트, Signed, 2의 보수)
    - 12~0 비트: 소수부 (13비트, 항상 양수)

    정수부는 내림하여 변환하고, 소수부를 정확한 2진 변환 방식으로 변환.
    """
    max_frac_value = 2**frac_bits  # 13비트 소수부 정밀도 (8192)

    # 1. 정수부와 소수부 정확하게 분리 (내림 적용)
    int_part = np.floor(value).astype(int)
    frac_part = abs(value - int_part)  # 소수부 절댓값 유지

    # 2. 정수부(7비트) 변환 (2의 보수 변환)
    if int_part < 0:
        int_binary = format((1 << int_bits) + int_part, f'0{int_bits}b')  # 7비트 2의 보수 변환
    else:
        int_binary = format(int_part, f'0{int_bits}b')  # 7비트 양수 표현

    # 3. 앞 12비트는 항상 0으로 유지
    int_binary = "0" * 12 + int_binary

    # 4. 소수부 변환 (부동소수점 오차 최소화)
    frac_binary = ""
    frac_value = frac_part  # 소수부 값 유지
    for _ in range(frac_bits):  # 13비트 반복
        frac_value *= 2
        if frac_value >= 1:
            frac_binary += "1"
            frac_value -= 1
        else:
            frac_binary += "0"

    # . 최종 32비트 바이너리 생성
    fixed_binary = int_binary + frac_binary
    fixed_binary_int = int(fixed_binary, 2)
    lower_20_bits = fixed_binary_int & 0xFFFFF
    return lower_20_bits

def signed_binary_to_int(binary_str):
    # 이진수의 길이
    n = len(binary_str)

    # 부호 비트 확인 (최상위 비트가 1이면 음수)
    if binary_str[0] == '1':
        # 2의 보수 방법으로 음수 변환
        return int(binary_str, 2) - (1 << n)
    else:
        # 양수는 그냥 변환
        return int(binary_str, 2)

def hex_to_binary(hex_str):
    """
    16비트 헥스 값을 16비트 바이너리 문자열로 변환하는 함수

    :param hex_str: 변환할 16비트 헥스 값 (예: "1A2F" 또는 0x1A2F)
    :return: 16비트 바이너리 문자열 (예: "0001101000101111")
    """
    # hex_str가 정수형일 경우 문자열로 변환
    if isinstance(hex_str, int):
        hex_str = format(hex_str, 'X')  # 16진수 대문자로 변환

    # 16진수를 10진수로 변환 후, 16비트 이진수로 변환
    binary_str = bin(int(hex_str, 16))[2:].zfill(16)
    return binary_str

def binary_to_decimal(binary_str):
    # 16비트 중 앞 3비트는 정수부, 뒤 13비트는 소수부
    int_part = int(binary_str[:3], 2)  # 정수부
    frac_part = int(binary_str[3:], 2) / (2 ** 13)  # 소수부를 2^13으로 나눔
    return int_part + frac_part  # 정수부와 소수부 합산





def top(data_list, tokens) : #20bit data

    assert len(data_list) == tokens
    binarized_inputs = [float_to_fixed_point(x) for x in data_list]

    # print("\n🧮 Binarized Inputs (Fixed-point 20-bit integers):")
    # for i, val in enumerate(binarized_inputs):
    #     print(f"{i:2}: {val:05X} ({val:020b})")  # 16진수와 20비트 이진수로 출력

    int_list            = [0] * tokens  # exp_int() - input
    exp_int_output      = [0] * tokens  # exp_int() - output
    fraction_list       = [0] * tokens  # cordic_exu() - input
    exp_fraction_output = [0] * tokens  # cordic_exu() - output
    exponential         = [0] * tokens  # exp(input) = exp(int) * exp(frac) -> and then truncated to 16-bit
    y_dividend          = [0] * tokens  # cordic diu input
    data_out            = [0] * tokens
    binary_values       = [0] * tokens
    int_values          = [0.0] * tokens


    ##### OBTAINING 'exp(integer part)' #####
    for i in range(tokens):
        int_list[i] = binarized_inputs[i] >> 13

    exp_int_output = exp_int(int_list)

    ##### OBTAINING 'exp(fraction part)' #####
    for i in range(tokens):
        fraction_list[i] = (binarized_inputs[i] & 0x1FFF) << 1  # 하위 13비트를 가져오고, LSB에 0을 추가(총 14bit 유효)

    exp_fraction_output = cordic_exu(fraction_list)


    for i in range(tokens):
       exp_whole = exp_int_output[i] * exp_fraction_output[i]  # 34-bit
       exponential[i] = (exp_whole >> 18) & 0xFFFF  # [33:18] 추출 (16-bit)

    exp_sum = sum(exponential)  # 23-bit
    x_divider = (exp_sum >> 8) & 0x7FFF  # {1'b0, exp_sum[22:8]} (16-bit)
    # print(f"🧮 x_divider → dec: {x_divider:5d}   bin: {format(x_divider, '016b')}   hex: {format(x_divider, '04X')}")

    for i in range (tokens):
        round_bit = (exponential[i] >> 7) & 1
        exp_trunc = (exponential[i] >> 8) & 0xFF
        exp_rounded = exp_trunc + 1 if round_bit else exp_trunc
        y_dividend_ = exp_rounded & 0x1FF
        y_dividend[i] = y_dividend_
        # print(f"🧮 y_dividend → dec: {y_dividend_:5d}   bin: {format(y_dividend_, '016b')}   hex: {format(y_dividend_, '04X')}")

    data_out = cordic_diu(x_divider, y_dividend)

    for i in range(tokens):
        binary_values[i] = hex_to_binary(data_out[i])
       # print(binary_values[i])

    for i in range(tokens):
        int_values[i] = binary_to_decimal(binary_values[i])
        #print(int_values[i])

    return int_values



def exp_int(int_inputs):
    tokens = len(int_inputs)
    int_data   = [0] * tokens
    int_output = [0] * tokens

    # 7-bit signed 변환
    for i in range(tokens):
        binary_str = bin(int_inputs[i])[2:].zfill(7)
        int_data[i] = signed_binary_to_int(binary_str)

    i_max = max(int_data)
    offset = 11 - i_max

    lookup = {
        0:  0b0000_0000_0000_0001_00,
        1:  0b0000_0000_0000_0010_11,
        2:  0b0000_0000_0000_0111_10,
        3:  0b0000_0000_0001_0100_00,
        4:  0b0000_0000_0011_0110_10,
        5:  0b0000_0000_1001_0100_10,
        6:  0b0000_0001_1001_0011_10,
        7:  0b0000_0100_0100_1000_11,
        8:  0b0000_1011_1010_0101_00,
        9:  0b0001_1111_1010_0111_00,
        10: 0b0101_0110_0000_1010_10,
        11: 0b1110_1001_1110_0010_01
    }

    for i in range(tokens):
        int_reg = int_data[i] + offset
        int_output[i] = lookup.get(int_reg, 0)

    return int_output



def cordic_exu (fraction_inputs):
    tokens = len(fraction_inputs)
    exp_fraction = [0] * tokens

    for i in range(tokens):
        exp_fraction[i]=  cordic_exu_pipe(fraction_inputs[i])
        #print(bin(data_list[i]))

    return exp_fraction

def cordic_exu_pipe(z_in):
    x_1,  y_1,  z_1  = exp_unit(0b1001101010001111, 0,    z_in, 0b0010001100100111, 1)
    x_2,  y_2,  z_2  = exp_unit(x_1,                y_1,  z_1,  0b0001000001011000, 2)
    x_3,  y_3,  z_3  = exp_unit(x_2,                y_2,  z_2,  0b0000100000001010, 3)
    x_4,  y_4,  z_4  = exp_unit(x_3,                y_3,  z_3,  0b0000010000000001, 4)
    x_5,  y_5,  z_5  = exp_unit(x_4,                y_4,  z_4,  0b0000001000000000, 5)
    x_6,  y_6,  z_6  = exp_unit(x_5,                y_5,  z_5,  0b0000000100000000, 6)
    x_7,  y_7,  z_7  = exp_unit(x_6,                y_6,  z_6,  0b0000000010000000, 7)
    x_8,  y_8,  z_8  = exp_unit(x_7,                y_7,  z_7,  0b0000000001000000, 8)
    x_9,  y_9,  z_9  = exp_unit(x_8,                y_8,  z_8,  0b0000000000100000, 9)
    x_10, y_10, z_10 = exp_unit(x_9,                y_9,  z_9,  0b0000000000010000, 10)
    x_11, y_11, z_11 = exp_unit(x_10,               y_10, z_10, 0b0000000000001000, 11)
    x_12, y_12, z_12 = exp_unit(x_11,               y_11, z_11, 0b0000000000000100, 12)
    x_13, y_13, z_13 = exp_unit(x_12,               y_12, z_12, 0b0000000000000010, 13)
    x_14, y_14, z_14 = exp_unit(x_13,               y_13, z_13, 0b0000000000000010, 14)

    exp_frac_reg = x_14 + y_14
    exp_frac = (exp_frac_reg >> 1) & 0xFFFF
    # print(bin(exp_frac))

    return exp_frac

def exp_unit(x_in, y_in, z_in, lut_val, pipe_num):
    # 16-bit 입력 값
    x_cal = 0
    y_cal = 0
    z_cal = 0

    if z_in < 0:  # z_in[15] = MSB를 확인 (시계방향: z_in[15]가 1)
        # 시계방향 (downward)
        x_cal = x_in - (y_in >> pipe_num)
        y_cal = y_in - (x_in >> pipe_num)
        z_cal = z_in + lut_val
    else:
        # 반시계방향 (upward)
        x_cal = x_in + (y_in >> pipe_num)
        y_cal = y_in + (x_in >> pipe_num)
        z_cal = z_in - lut_val

    return x_cal, y_cal, z_cal



def cordic_diu (x_divider_in, y_dividend_in):
    tokens = len(y_dividend_in)
    softmax = [0] * tokens

    for i in range(tokens):
        softmax[i]=  cordic_diu_pipe(x_divider_in, y_dividend_in[i])

    return softmax

def cordic_diu_pipe(x_in, y_in):
    y_1,  z_1  = div_unit(x_in, y_in,   0,  0b0010000000000000, 0)
    y_2,  z_2  = div_unit(x_in, y_1,  z_1,  0b0001000000000000, 1)
    y_3,  z_3  = div_unit(x_in, y_2,  z_2,  0b0000100000000000, 2)
    y_4,  z_4  = div_unit(x_in, y_3,  z_3,  0b0000010000000000, 3)
    y_5,  z_5  = div_unit(x_in, y_4,  z_4,  0b0000001000000000, 4)
    y_6,  z_6  = div_unit(x_in, y_5,  z_5,  0b0000000100000000, 5)
    y_7,  z_7  = div_unit(x_in, y_6,  z_6,  0b0000000010000000, 6)
    y_8,  z_8  = div_unit(x_in, y_7,  z_7,  0b0000000001000000, 7)
    y_9,  z_9  = div_unit(x_in, y_8,  z_8,  0b0000000000100000, 8)
    y_10, z_10 = div_unit(x_in, y_9,  z_9,  0b0000000000010000, 9)
    y_11, z_11 = div_unit(x_in, y_10, z_10, 0b0000000000001000, 10)
    y_12, z_12 = div_unit(x_in, y_11, z_11, 0b0000000000000100, 11)
    y_13, z_13 = div_unit(x_in, y_12, z_12, 0b0000000000000010, 12)
    y_14, z_14 = div_unit(x_in, y_13, z_13, 0b0000000000000001, 13)

    divided_val = z_14 & 0xFFFF # Final z value representing atan(y/x)

    # print(f"🧮 CORDIC_DIV Output → dec: {divided_val:5d}   bin: {format(divided_val, '016b')}   hex: {format(divided_val, '04X')}")

    return divided_val

def div_unit(x_in, y_in, z_in, lut_val, pipe_num):
    # 16-bit 입력 값
    y_cal = 0
    z_cal = 0

    if y_in < 0:  # z_in[15] = MSB를 확인 (시계방향: z_in[15]가 1)
        y_cal = y_in + (x_in >> pipe_num)
        z_cal = z_in - lut_val
    else:
        y_cal = y_in - (x_in >> pipe_num)
        z_cal = z_in + lut_val

    return  y_cal, z_cal



######################################################################
########################### ↓↓↓ TEST ↓↓↓ #############################
######################################################################

if __name__ == "__main__":
    import numpy as np

    # 16개 입력값 생성 (예: -5 ~ 5 사이의 균일한 분포)
    tokens = 128
    # np.random.seed(42)
    # input_values = np.round(np.random.uniform(-5, 5, tokens), 6)  # 고정소수점 정밀도 감안해서 소수점 제한
    # input_values = [0.6330924, 0.4353165, -0.2769869, -0.47311664, -0.6537097, 1.1346276, 1.2043934, 1.2259121,
    #                 2.4453104, 3.447215, 2.2860608, 1.9619465, 1.677911, 1.6217171, 1.0423713, 2.3396838,
    #                 2.8496568, 3.0143862, 3.1036558, 3.580126, 2.176156, 2.079998, 1.9142365, 1.2545184,
    #                 0.90532553, 2.4406319, 2.6811087, 2.771984, 3.706369, 4.3801403, 2.7754946, 2.54271,
    #                 2.0250046, 1.7798194, 1.6800123, 3.0317712, 3.0615597, 3.1782565, 3.5403452, 3.9254875,
    #                 2.8844366, 2.758453, 2.2565348, 1.232966, 1.2297448, 3.3585246, 3.6153154, 4.230633,
    #                 4.4068875, 4.697502, 2.819391, 2.5985827, 2.5125113, 2.3833961, 1.9145572, 3.207824,
    #                 3.3063452, 3.395969, 3.6085682, 3.813313, 4.5458984, 4.1019, 3.9767156, 3.8136652,
    #                 0.6330924, 0.4353165, -0.2769869, -0.47311664, -0.6537097, 1.1346276, 1.2043934, 1.2259121,
    #                 2.4453104, 3.447215, 2.2860608, 1.9619465, 1.677911, 1.6217171, 1.0423713, 2.3396838,
    #                 2.8496568, 3.0143862, 3.1036558, 3.580126, 2.176156, 2.079998, 1.9142365, 1.2545184,
    #                 0.90532553, 2.4406319, 2.6811087, 2.771984, 3.706369, 4.3801403, 2.7754946, 2.54271,
    #                 2.0250046, 1.7798194, 1.6800123, 3.0317712, 3.0615597, 3.1782565, 3.5403452, 3.9254875,
    #                 2.8844366, 2.758453, 2.2565348, 1.232966, 1.2297448, 3.3585246, 3.6153154, 4.230633,
    #                 4.4068875, 4.697502, 2.819391, 2.5985827, 2.5125113, 2.3833961, 1.9145572, 3.207824,
    #                 3.3063452, 3.395969, 3.6085682, 3.813313, 4.5458984, 4.1019, 3.9767156, 3.8136652]
    input_values = [1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999,
                    1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999,
                    1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999,
                    1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999,
                    1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999,
                    1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999,
                    1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999,
                    1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999, 1.9999]


    print("🔢 Input Values (float):")
    # print(input_values.tolist())
    print(input_values)

    # 모델 실행
    # softmax_outputs = top(input_values.tolist(), tokens)
    softmax_outputs = top(input_values, tokens)

    print("\n📤 Softmax Output Values (float):")
    for i, val in enumerate(softmax_outputs):
        scaled_val = int(round(val * (1 << 13)))  # Q3.13 고정소수점 정수 표현
        print(f"{i:2}: {val:.6f}   (×2^13 → {scaled_val:5d})")

    # 합이 1에 근접하는지 확인
    softmax_sum = sum(softmax_outputs)
    print(f"\n✅ Softmax Sum = {softmax_sum:.6f}")

### 구조 적용,, 재시작

In [ ]:
import torch
from transformers.models.bert.modeling_bert import BertSelfAttention

class BertSelfAttentionModified(BertSelfAttention):
    def forward(self,
                hidden_states,
                attention_mask=None,
                head_mask=None,
                encoder_hidden_states=None,
                encoder_attention_mask=None,
                past_key_value=None,
                output_attentions=False):

        # 1) query/key/value (원본과 동일)
        mixed_query_layer = self.query(hidden_states)
        key_layer   = self.transpose_for_scores(
                          self.key(encoder_hidden_states
                                   if encoder_hidden_states is not None
                                   else hidden_states)
                      )
        value_layer = self.transpose_for_scores(
                          self.value(encoder_hidden_states
                                     if encoder_hidden_states is not None
                                     else hidden_states)
                      )
        query_layer = self.transpose_for_scores(mixed_query_layer)

        # 2) 스코어 계산 및 scaling
        attention_scores = torch.matmul(
            query_layer, key_layer.transpose(-1, -2)
        ) / (self.attention_head_size ** 0.5)
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        # 3) CORDIC-Softmax 교체

        B, H, L, _ = attention_scores.size()           # e.g. 1,12,10,10
        flat = attention_scores.view(-1, L)            # [B*H, L]
        rows = []

       # for row in flat:
       #     # (1) 안정화: max 빼기
       #     #row = row - row.max()
       #     # (2) 정수부 범위 클램핑 (–63.999 ~ +63.999)
       #     row = row.clamp(min=-63.999, max=63.999)
       #     # (3) Python float 리스트로 변환 → top()
       #     out = top(*row.tolist())
       #     # (4) 다시 tensor 생성
       #     rows.append(torch.tensor(
       #         out,
       #         dtype=attention_scores.dtype,
       #         device=attention_scores.device
       #     ))
        for row in flat:
            row = row.clamp(min=-63.999, max=63.999)
            row_list = row.tolist()
            out = top(row_list, L)
            rows.append(torch.tensor(
                out, dtype=attention_scores.dtype, device=attention_scores.device
            ))

        attention_probs = torch.stack(rows).view(B, H, L, L)
        # ─────────────────────────────────────────────

        # 4) dropout & context 계산 (원본과 동일)
        attention_probs = self.dropout(attention_probs)
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_shape)

        outputs = (context_layer, attention_probs) \
                  if output_attentions else (context_layer,)
        return outputs


In [ ]:
config = model.config
for layer in model.bert.encoder.layer:
    orig = layer.attention.self
    # Modified 모듈 생성
    mod = BertSelfAttentionModified(config)
    # 기존 가중치 로드
    mod.load_state_dict(orig.state_dict(), strict=False)

    mod.to(device)
    # 교체
    layer.attention.self = mod


In [ ]:
import inspect
for idx, layer in enumerate(model.bert.encoder.layer):
    attn = layer.attention.self
    print(f"Layer {idx:2d}: {attn.__class__.__name__:<30} (module: {attn.__class__.__module__})")

print("모델 클래스:", model.__class__.__name__)
# → BertForSequenceClassificationCordic 이어야 함

# 2) forward 메서드가 서브클래스에서 정의된 것인지 확인
print("forward 메서드 qualname:", model.forward.__qualname__)
# → BertForSequenceClassificationCordic.forward 여야 함

# 3) 실제 forward 소스 코드 출력
source = inspect.getsource(model.__class__.forward)
print("\n----- forward 소스 -----\n", source)

In [ ]:
# 1) forward pass
model.eval()
with torch.no_grad():
    outputs = model(
        b_input_ids,
        attention_mask=b_input_mask,
        output_attentions=True
    )

# 2) 튜플에서 마지막 레이어 확률만 꺼내기
all_attentions = outputs.attentions    # tuple of length 12, each is [B, H, L, L]
attention_probs = all_attentions[-1]   # 마지막 레이어의 [B, H, L, L] 텐서

# 3) 마지막 축(키 방향)으로 합이 1인지 확인
sums = attention_probs.sum(dim=-1)     # shape = [B, H, L]
print("어텐션 확률 합:", sums)


In [ ]:
import time

def run_validation(model, validation_dataloader, device):
    print("== Self-Attention Modules ==")
    for idx, layer in enumerate(model.bert.encoder.layer):
        cls_name = layer.attention.self.__class__.__name__
        print(f"  Layer {idx:2d}: {cls_name}")
    print("=============================\n")
    model.eval()
    eval_accuracy = 0.0
    nb_eval_steps = 0
    t0 = time.time()

    with torch.no_grad():
        for batch in validation_dataloader:
            # 배치를 GPU로
            b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

            #logits만 얻기
            outputs = model(
                b_input_ids,
                token_type_ids=None,
                attention_mask=b_input_mask
            )
            logits = outputs[0]

            # CPU로 내리고 numpy 변환
            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.cpu().numpy()

            # 배치별 정확도 계산
            batch_acc = flat_accuracy(logits, label_ids)
            eval_accuracy += batch_acc
            nb_eval_steps += 1

    avg_accuracy = eval_accuracy / nb_eval_steps
    print(f"Validation Accuracy: {avg_accuracy:.2f}")
    print(f"Validation Time   : {format_time(time.time() - t0)}")

    return avg_accuracy

avg_acc = run_validation(model, validation_dataloader, device)

In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import (
    matthews_corrcoef,
    precision_recall_fscore_support,
    confusion_matrix
)

def run_validation(model, validation_dataloader, device):
    # 1) Self-Attention 모듈 확인
    print("== Self-Attention Modules ==")
    for idx, layer in enumerate(model.bert.encoder.layer):
        cls_name = layer.attention.self.__class__.__name__
        print(f"  Layer {idx:2d}: {cls_name}")
    print("=============================\n")

    # 2) 준비
    model.eval()
    criterion = nn.CrossEntropyLoss()
    eval_accuracy = 0.0
    eval_loss = 0.0
    nb_eval_steps = 0

    all_preds = []
    all_labels = []

    t0 = time.time()

    with torch.no_grad():
        for batch in validation_dataloader:
            b_input_ids, b_input_mask, b_labels = [t.to(device) for t in batch]

            # Forward pass (logits 얻기)
            outputs = model(
                b_input_ids,
                token_type_ids=None,
                attention_mask=b_input_mask
            )
            logits = outputs[0]              # [B, 2]

            # Loss 계산
            loss = criterion(logits, b_labels)
            eval_loss += loss.item()

            # Accuracy
            preds = np.argmax(logits.detach().cpu().numpy(), axis=1)
            #probs_tensor = torch.softmax(logits, dim=1)               # [B,2]
            #pos_probs = probs_tensor[:, 1].detach().cpu().numpy()     # positive 클래스 확률
            #threshold = 0.6                                           # 원하는 임계치
            #preds = (pos_probs > threshold).astype(int)               # 0.6 초과면 1, 아니면 0
            label_ids = b_labels.detach().cpu().numpy()
            batch_acc = np.sum(preds == label_ids) / len(label_ids)
            eval_accuracy += batch_acc

            # 저장
            all_preds.extend(preds.tolist())
            all_labels.extend(label_ids.tolist())

            nb_eval_steps += 1

    # 3) 평균 지표 계산
    avg_accuracy = eval_accuracy / nb_eval_steps
    avg_loss     = eval_loss / nb_eval_steps
    mcc          = matthews_corrcoef(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )
    cm = confusion_matrix(all_labels, all_preds)

    # 4) 결과 출력
    print(f"Validation Loss    : {avg_loss:.4f}")
    print(f"Validation Accuracy: {avg_accuracy:.4f}")
    print(f"Matthews CorrCoef  : {mcc:.4f}")
    print(f"Precision / Recall / F1: {precision:.4f} / {recall:.4f} / {f1:.4f}")
    print("Confusion Matrix:\n", cm)
    print(f"Validation Time    : {format_time(time.time() - t0)}")

    return {
        "accuracy": avg_accuracy,
        "loss": avg_loss,
        "mcc": mcc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm
    }

# 사용 예시
metrics = run_validation(model, validation_dataloader, device)


In [ ]:
print(L)

## STS-B all

### STS-B

In [ ]:
import tensorflow as tf

# Get the GPU device name.
device_name = tf.test.gpu_device_name()

# The device name should look like the following:
if device_name == '/device:GPU:0':
    print('Found GPU at: {}'.format(device_name))
else:
    raise SystemError('GPU device not found')

import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
!pip install transformers
!pip install wget
!pip install datasets
!pip install scikit-learn
!pip install keras
!pip install --upgrade sympy
!pip install --upgrade fsspec datasets
import os
import wget
import pandas as pd
import torch
import numpy as np
import time, datetime

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

In [ ]:
!pip install evaluate
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Same as before
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=1,                # 회귀용 출력 하나
    problem_type="regression"    # 회귀 문제로 지정
)
sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# This is new
batch["labels"] = torch.tensor([1.0, 1.0], dtype=torch.float32)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,       # 권장 2e-5 ∼ 5e-5 범위
    weight_decay=0.01
)
loss = model(**batch).loss
loss.backward()
optimizer.step()

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "stsb")
tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer)

def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        max_length=8  #################################################################################################################################################################################################
    )

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# 확인
print(tokenized_datasets["train"].column_names)
# → ['input_ids', 'token_type_ids', 'attention_mask', 'labels']

In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=64, collate_fn=data_collator
)

for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=1,                # STS-B는 회귀이므로 출력 하나
    problem_type="regression"    # 회귀 문제로 지정
)

outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5    # STS-B 권장 기본 learning rate
)

from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)
print(num_training_steps)

import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device

In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
model.to(device)

In [ ]:
import torch
from tqdm.auto import tqdm
import evaluate

# 1) metric 로드
metric = evaluate.load("glue", "stsb")

model.eval()

with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating", unit="batch"):
        # 2) labels는 CPU에, inputs만 GPU로 옮기기
        labels = batch["labels"]                    # (batch, ) torch.Tensor on CPU
        inputs = {
            k: v.to(device)
            for k, v in batch.items()
            if k != "labels"
        }

        # 3) labels 없이 모델에 입력 → logits만 리턴
        outputs = model(**inputs)
        logits = outputs.logits                     # (batch, num_labels) on GPU

        # 4) 예측만 CPU로 가져오기
        preds = logits.squeeze(-1).cpu()   # (batch,)

        # 5) metric에 추가
        metric.add_batch(predictions=preds, references=labels)

# 6) 최종 지표 출력
print(metric.compute())


### STS-B 구조 적용

In [ ]:
import torch
from transformers.models.bert.modeling_bert import BertSelfAttention

class BertSelfAttentionModified(BertSelfAttention):
    def forward(self,
                hidden_states,
                attention_mask=None,
                head_mask=None,
                encoder_hidden_states=None,
                encoder_attention_mask=None,
                past_key_value=None,
                output_attentions=False):

        # 1) query/key/value (원본과 동일)
        mixed_query_layer = self.query(hidden_states)
        key_layer   = self.transpose_for_scores(
                          self.key(encoder_hidden_states
                                   if encoder_hidden_states is not None
                                   else hidden_states)
                      )
        value_layer = self.transpose_for_scores(
                          self.value(encoder_hidden_states
                                     if encoder_hidden_states is not None
                                     else hidden_states)
                      )
        query_layer = self.transpose_for_scores(mixed_query_layer)

        # 2) 스코어 계산 및 scaling
        attention_scores = torch.matmul(
            query_layer, key_layer.transpose(-1, -2)
        ) / (self.attention_head_size ** 0.5)
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        # 3) CORDIC-Softmax 교체

        B, H, L, _ = attention_scores.size()           # e.g. 1,12,10,10
        flat = attention_scores.view(-1, L)            # [B*H, L]
        rows = []

       # for row in flat:
       #     # (1) 안정화: max 빼기
       #     #row = row - row.max()
       #     # (2) 정수부 범위 클램핑 (–63.999 ~ +63.999)
       #     row = row.clamp(min=-63.999, max=63.999)
       #     # (3) Python float 리스트로 변환 → top()
       #     out = top(*row.tolist())
       #     # (4) 다시 tensor 생성
       #     rows.append(torch.tensor(
       #         out,
       #         dtype=attention_scores.dtype,
       #         device=attention_scores.device
       #     ))
        for row in flat:
            row = row.clamp(min=-63.999, max=63.999)
            row_list = row.tolist()
            out = top(row_list, L)
            rows.append(torch.tensor(
                out, dtype=attention_scores.dtype, device=attention_scores.device
            ))

        attention_probs = torch.stack(rows).view(B, H, L, L)
        # ─────────────────────────────────────────────

        # 4) dropout & context 계산 (원본과 동일)
        attention_probs = self.dropout(attention_probs)
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_shape)

        outputs = (context_layer, attention_probs) \
                  if output_attentions else (context_layer,)
        return outputs

config = model.config
for layer in model.bert.encoder.layer:
    orig = layer.attention.self
    # Modified 모듈 생성
    mod = BertSelfAttentionModified(config)
    # 기존 가중치 로드
    mod.load_state_dict(orig.state_dict(), strict=False)

    mod.to(device)
    # 교체
    layer.attention.self = mod


In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
from tqdm.auto import tqdm
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score

def run_validation_stsb(model, validation_dataloader, device):
    # 1) Self-Attention 모듈 확인
    print("== Self-Attention Modules (STS-B) ==")
    for idx, layer in enumerate(model.bert.encoder.layer):
        cls_name = layer.attention.self.__class__.__name__
        print(f"  Layer {idx:2d}: {cls_name}")
    print("===============================\n")

    # 2) device에 올리고 eval 모드
    model.to(device)
    model.eval()

    # 손실 함수 정의
    mse_fn = nn.MSELoss()    # MSE
    mae_fn = nn.L1Loss()     # MAE

    total_mse = 0.0
    total_mae = 0.0
    steps = 0

    all_preds = []
    all_labels = []

    start = time.time()
    for batch in tqdm(validation_dataloader, desc="Validating STS-B"):
        # 배치 내 모든 텐서를 device로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # inputs 구성
        inputs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        }
        if "token_type_ids" in batch:
            inputs["token_type_ids"] = batch["token_type_ids"]

        # 레이블 float 변환
        labels = batch["labels"].to(torch.float32)

        # forward (회귀)
        with torch.no_grad():
            outputs = model(**inputs)
        preds = outputs.logits.squeeze(-1)  # [B]

        # 누적 손실
        total_mse += mse_fn(preds, labels).item()
        total_mae += mae_fn(preds, labels).item()
        steps += 1

        # 예측값/정답 수집
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    # 평균 & 추가 지표 계산
    avg_mse  = total_mse / steps
    avg_mae  = total_mae / steps
    rmse     = np.sqrt(avg_mse)
    pearson  = pearsonr(all_labels, all_preds)[0]
    spearman = spearmanr(all_labels, all_preds)[0]
    elapsed  = time.time() - start

    # 결과 출력
    print(f"\nValidation MSE      : {avg_mse:.4f}")
    print(f"Validation RMSE     : {rmse:.4f}")
    print(f"Validation MAE      : {avg_mae:.4f}")
    print(f"Pearson Correlation : {pearson:.4f}")
    print(f"Spearman Correlation: {spearman:.4f}")
    print(f"Validation Time     : {elapsed:.1f}s\n")

    return {
        "mse": avg_mse,
        "rmse": rmse,
        "mae": avg_mae,
        "pearson": pearson,
        "spearman": spearman,
    }

# 사용 예시
metrics_stsb = run_validation_stsb(model, eval_dataloader, device)


## MNLI all

### MNLI

In [ ]:
import tensorflow as tf

# Get the GPU device name.
device_name = tf.test.gpu_device_name()

# The device name should look like the following:
if device_name == '/device:GPU:0':
    print('Found GPU at: {}'.format(device_name))
else:
    raise SystemError('GPU device not found')

import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
!pip install --upgrade sympy
!pip install --upgrade fsspec datasets
!pip install transformers
!pip install wget
!pip install datasets
!pip install evaluate
!pip install scikit-learn
!pip install keras
import os
import wget
import pandas as pd
import torch
import numpy as np
import time, datetime

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.optim import AdamW

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=3  # MNLI는 3-class 분류
)

sequences = [
    "A soccer game with multiple males playing.",
    "Some men are playing a sport.",
]
batch = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# MNLI용 분류 레이블 (예: entailment=0, neutral=1, contradiction=2)
batch["labels"] = torch.tensor([0, 0], dtype=torch.long)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss = model(**batch).loss
loss.backward()
optimizer.step()


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", "mnli")
tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer)

def tokenize_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        max_length=64    #################################################################################################################################################################################################
    )

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["premise", "hypothesis", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# 확인
print(tokenized_datasets["train"].column_names)
# → ['input_ids', 'token_type_ids', 'attention_mask', 'labels']

In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["validation_matched"], batch_size=64, collate_fn=data_collator
)

mismatched_dataloader = DataLoader(
    tokenized_datasets["validation_mismatched"],
    batch_size=64,
    collate_fn=data_collator
)

for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=3,  # entailment / neutral / contradiction
)
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)  # logits: (batch_size, 3)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,            # ✅ GLUE 논문 수준 MNLI 학습률
    weight_decay=0.01   # ✅ 논문에서 사용하는 L2 정규화
)

from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",                                 # ✅ GLUE 기본 스케줄러
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),  # ✅ 10% warmup
    num_training_steps=num_training_steps,
)
print(num_training_steps)

import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device


In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
model.to(device)


In [ ]:
import torch
from tqdm.auto import tqdm
import evaluate

# 1) 두 개의 metric 객체 생성
metric_matched   = evaluate.load("glue", "mnli")
metric_mismatched = evaluate.load("glue", "mnli")

model.eval()

# 2) validation_matched 평가
for batch in tqdm(eval_dataloader, desc="Evaluating matched", unit="batch"):
    labels = batch["labels"]
    inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    preds  = torch.argmax(logits, dim=-1).cpu()

    metric_matched.add_batch(predictions=preds, references=labels)

results_matched = metric_matched.compute()["accuracy"]

# 3) validation_mismatched 평가
for batch in tqdm(mismatched_dataloader, desc="Evaluating mismatched", unit="batch"):
    labels = batch["labels"]
    inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}

    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    preds  = torch.argmax(logits, dim=-1).cpu()

    metric_mismatched.add_batch(predictions=preds, references=labels)

results_mismatched = metric_mismatched.compute()["accuracy"]

# 4) 결과 출력 및 평균 계산
print(f"Matched Accuracy   : {results_matched:.2%}")
print(f"Mismatched Accuracy: {results_mismatched:.2%}")
print(f"Average Accuracy   : {((results_matched + results_mismatched) / 2):.2%}")


### MNLI 구조 적용


In [ ]:
import torch
from transformers.models.bert.modeling_bert import BertSelfAttention

class BertSelfAttentionModified(BertSelfAttention):
    def forward(self,
                hidden_states,
                attention_mask=None,
                head_mask=None,
                encoder_hidden_states=None,
                encoder_attention_mask=None,
                past_key_value=None,
                output_attentions=False):

        # 1) query/key/value (원본과 동일)
        mixed_query_layer = self.query(hidden_states)
        key_layer   = self.transpose_for_scores(
                          self.key(encoder_hidden_states
                                   if encoder_hidden_states is not None
                                   else hidden_states)
                      )
        value_layer = self.transpose_for_scores(
                          self.value(encoder_hidden_states
                                     if encoder_hidden_states is not None
                                     else hidden_states)
                      )
        query_layer = self.transpose_for_scores(mixed_query_layer)

        # 2) 스코어 계산 및 scaling
        attention_scores = torch.matmul(
            query_layer, key_layer.transpose(-1, -2)
        ) / (self.attention_head_size ** 0.5)
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        # 3) CORDIC-Softmax 교체

        B, H, L, _ = attention_scores.size()           # e.g. 1,12,10,10
        flat = attention_scores.view(-1, L)            # [B*H, L]
        rows = []

       # for row in flat:
       #     # (1) 안정화: max 빼기
       #     #row = row - row.max()
       #     # (2) 정수부 범위 클램핑 (–63.999 ~ +63.999)
       #     row = row.clamp(min=-63.999, max=63.999)
       #     # (3) Python float 리스트로 변환 → top()
       #     out = top(*row.tolist())
       #     # (4) 다시 tensor 생성
       #     rows.append(torch.tensor(
       #         out,
       #         dtype=attention_scores.dtype,
       #         device=attention_scores.device
       #     ))
        for row in flat:
            row = row.clamp(min=-63.999, max=63.999)
            row_list = row.tolist()
            out = top(row_list, L)
            rows.append(torch.tensor(
                out, dtype=attention_scores.dtype, device=attention_scores.device
            ))

        attention_probs = torch.stack(rows).view(B, H, L, L)
        # ─────────────────────────────────────────────

        # 4) dropout & context 계산 (원본과 동일)
        attention_probs = self.dropout(attention_probs)
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_shape)

        outputs = (context_layer, attention_probs) \
                  if output_attentions else (context_layer,)
        return outputs

config = model.config
for layer in model.bert.encoder.layer:
    orig = layer.attention.self
    # Modified 모듈 생성
    mod = BertSelfAttentionModified(config)
    # 기존 가중치 로드
    mod.load_state_dict(orig.state_dict(), strict=False)

    mod.to(device)
    # 교체
    layer.attention.self = mod


In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (
    matthews_corrcoef,
    precision_recall_fscore_support,
    confusion_matrix
)

def run_validation(model, validation_dataloader, device, split_name="validation"):
    # 1) Self-Attention 모듈 확인 (한 번만 출력해도 OK)
    print(f"== Self-Attention Modules ({split_name}) ==")
    for idx, layer in enumerate(model.bert.encoder.layer):
        cls_name = layer.attention.self.__class__.__name__
        print(f"  Layer {idx:2d}: {cls_name}")
    print("=" * 30 + "\n")

    # 2) device 올리고 eval 모드
    model.to(device)
    model.eval()

    criterion     = nn.CrossEntropyLoss()
    eval_loss     = 0.0
    nb_eval_steps = 0

    all_preds  = []
    all_labels = []

    t0 = time.time()
    for batch in tqdm(validation_dataloader, desc=f"Validating {split_name}"):
        # 배치 내 모든 텐서를 device로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # forward
        with torch.no_grad():
            outputs = model(
                input_ids      = batch["input_ids"],
                token_type_ids = batch.get("token_type_ids", None),
                attention_mask = batch["attention_mask"]
            )
        logits = outputs.logits  # [B, num_labels]

        # loss 누적
        loss = criterion(logits, batch["labels"])
        eval_loss += loss.item()
        nb_eval_steps += 1

        # 예측값 & 정답 수집
        preds  = np.argmax(logits.cpu().numpy(), axis=1)
        labels = batch["labels"].cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    # 평균 지표 계산 (multi-class → macro 평균)
    avg_loss  = eval_loss / nb_eval_steps
    avg_acc   = np.mean(np.array(all_preds) == np.array(all_labels))
    mcc       = matthews_corrcoef(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro'
    )
    cm = confusion_matrix(all_labels, all_preds)
    elapsed = time.time() - t0

    # 결과 출력
    print(f"\n[{split_name}] Loss     : {avg_loss:.4f}")
    print(f"[{split_name}] Accuracy : {avg_acc:.4f}")
    print(f"[{split_name}] MCC      : {mcc:.4f}")
    print(f"[{split_name}] Precision: {precision*100:.2f}%")
    print(f"[{split_name}] Recall   : {recall*100:.2f}%")
    print(f"[{split_name}] F1-score : {f1*100:.2f}%")
    print(f"[{split_name}] Confusion Matrix:\n{cm}")
    print(f"[{split_name}] Time     : {elapsed:.1f}s\n")

    return {
        "loss": avg_loss,
        "accuracy": avg_acc,
        "mcc": mcc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
    }

# ── 사용 예시 (MNLI)
metrics_matched    = run_validation(model, eval_dataloader,          device, split_name="matched")
metrics_mismatched = run_validation(model, mismatched_dataloader,    device, split_name="mismatched")

# 평균 정확도 계산
avg_accuracy = (metrics_matched["accuracy"] + metrics_mismatched["accuracy"]) / 2
print(f"Average Accuracy : {avg_accuracy:.2%}")


## QNLI all

### QNLI

In [ ]:
import tensorflow as tf

# Get the GPU device name.
device_name = tf.test.gpu_device_name()

# The device name should look like the following:
if device_name == '/device:GPU:0':
    print('Found GPU at: {}'.format(device_name))
else:
    raise SystemError('GPU device not found')

import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
!pip install --upgrade sympy
!pip install --upgrade fsspec datasets
!pip install transformers
!pip install wget
!pip install datasets
!pip install scikit-learn
!pip install keras
!pip install evaluate
import os
import wget
import pandas as pd
import torch
import numpy as np
import time, datetime

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.optim import AdamW

checkpoint = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(checkpoint)
model      = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2   # QNLI는 2-class 분류
)

# 1) QNLI용 예시: (question, sentence) 쌍
sequences = [
    ("What causes rain?", "Rain is caused by water vapor condensing in the atmosphere."),
    ("Who wrote 'Pride and Prejudice'?", "Jane Austen wrote the novel Pride and Prejudice."),
]
batch = tokenizer(
    [q for q, s in sequences],
    [s for q, s in sequences],
    padding=True, truncation=True, return_tensors="pt"
)

# 2) 이진 레이블: entailment=1, non-entailment=0
batch["labels"] = torch.tensor([1, 1], dtype=torch.long)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,         # GLUE 논문 권장 범위
    weight_decay=0.01
)

# 3) forward + backward
outputs = model(**batch)
loss = outputs.loss
loss.backward()
optimizer.step()


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# 1) QNLI 데이터셋 로드
raw_datasets = load_dataset("glue", "qnli")
tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer)

# 2) tokenize 함수: question ↔ sentence 쌍
def tokenize_function(examples):
    return tokenizer(
        examples["question"],
        examples["sentence"],
        truncation=True,
        max_length=64    #================================================================================================================================================================
    )

# 3) 토크나이징 & 컬럼 정리
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["question", "sentence", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# 4) 확인
print(tokenized_datasets["train"].column_names)
# → ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=64, collate_fn=data_collator
)

for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
)
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)  # logits: (batch_size, 3)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,            # GLUE 논문 권장 learning rate (2e-5 ~ 5e-5)
    weight_decay=0.01   # GLUE 논문 권장 weight decay
)

from transformers import get_scheduler

num_epochs = 3                                # GLUE 기본 epoch 수
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",                                  # GLUE 논문 기본 scheduler (linear decay)
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),  # GLUE 논문 권장 warmup 10%
    num_training_steps=num_training_steps,
)
print(num_training_steps)

import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device


In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
model.to(device)


In [ ]:
import evaluate
from tqdm.auto import tqdm

metric = evaluate.load("glue", "qnli")
model.eval()

with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating", unit="batch"):
        labels = batch["labels"]
        inputs = {k: v.to(device) for k,v in batch.items() if k!="labels"}

        outputs = model(**inputs)
        logits  = outputs.logits       # (batch, 2)
        preds   = torch.argmax(logits, dim=-1).cpu()

        metric.add_batch(predictions=preds, references=labels)

print(metric.compute())  # {'accuracy': …}


### QNLI 구조 적용

In [ ]:
import torch
from transformers.models.bert.modeling_bert import BertSelfAttention

class BertSelfAttentionModified(BertSelfAttention):
    def forward(self,
                hidden_states,
                attention_mask=None,
                head_mask=None,
                encoder_hidden_states=None,
                encoder_attention_mask=None,
                past_key_value=None,
                output_attentions=False):

        # 1) query/key/value (원본과 동일)
        mixed_query_layer = self.query(hidden_states)
        key_layer   = self.transpose_for_scores(
                          self.key(encoder_hidden_states
                                   if encoder_hidden_states is not None
                                   else hidden_states)
                      )
        value_layer = self.transpose_for_scores(
                          self.value(encoder_hidden_states
                                     if encoder_hidden_states is not None
                                     else hidden_states)
                      )
        query_layer = self.transpose_for_scores(mixed_query_layer)

        # 2) 스코어 계산 및 scaling
        attention_scores = torch.matmul(
            query_layer, key_layer.transpose(-1, -2)
        ) / (self.attention_head_size ** 0.5)
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        # 3) CORDIC-Softmax 교체

        B, H, L, _ = attention_scores.size()           # e.g. 1,12,10,10
        flat = attention_scores.view(-1, L)            # [B*H, L]
        rows = []

       # for row in flat:
       #     # (1) 안정화: max 빼기
       #     #row = row - row.max()
       #     # (2) 정수부 범위 클램핑 (–63.999 ~ +63.999)
       #     row = row.clamp(min=-63.999, max=63.999)
       #     # (3) Python float 리스트로 변환 → top()
       #     out = top(*row.tolist())
       #     # (4) 다시 tensor 생성
       #     rows.append(torch.tensor(
       #         out,
       #         dtype=attention_scores.dtype,
       #         device=attention_scores.device
       #     ))
        for row in flat:
            row = row.clamp(min=-63.999, max=63.999)
            row_list = row.tolist()
            out = top(row_list, L)
            rows.append(torch.tensor(
                out, dtype=attention_scores.dtype, device=attention_scores.device
            ))

        attention_probs = torch.stack(rows).view(B, H, L, L)
        # ─────────────────────────────────────────────

        # 4) dropout & context 계산 (원본과 동일)
        attention_probs = self.dropout(attention_probs)
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_shape)

        outputs = (context_layer, attention_probs) \
                  if output_attentions else (context_layer,)
        return outputs

config = model.config
for layer in model.bert.encoder.layer:
    orig = layer.attention.self
    # Modified 모듈 생성
    mod = BertSelfAttentionModified(config)
    # 기존 가중치 로드
    mod.load_state_dict(orig.state_dict(), strict=False)

    mod.to(device)
    # 교체
    layer.attention.self = mod


In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (
    matthews_corrcoef,
    precision_recall_fscore_support,
    confusion_matrix
)

def run_validation_qnli(model, validation_dataloader, device):
    # 1) Self-Attention 모듈 확인
    print("== Self-Attention Modules (QNLI) ==")
    for idx, layer in enumerate(model.bert.encoder.layer):
        cls_name = layer.attention.self.__class__.__name__
        print(f"  Layer {idx:2d}: {cls_name}")
    print("===============================\n")

    # 2) device에 올리고 eval 모드
    model.to(device)
    model.eval()

    criterion     = nn.CrossEntropyLoss()
    eval_loss     = 0.0
    nb_eval_steps = 0

    all_preds  = []
    all_labels = []

    t0 = time.time()
    for batch in tqdm(validation_dataloader, desc="Validating QNLI"):
        # 3) batch를 device로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # 4) forward
        with torch.no_grad():
            outputs = model(
                input_ids      = batch["input_ids"],
                token_type_ids = batch["token_type_ids"],
                attention_mask = batch["attention_mask"]
            )
        logits = outputs.logits  # [B, 2]

        # 5) loss 누적
        loss = criterion(logits, batch["labels"])
        eval_loss += loss.item()
        nb_eval_steps += 1

        # 6) preds & labels 수집
        preds  = np.argmax(logits.cpu().numpy(), axis=1)
        labels = batch["labels"].cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    # 7) 평균 지표 계산
    avg_loss  = eval_loss / nb_eval_steps
    avg_acc   = np.mean(np.array(all_preds) == np.array(all_labels))
    mcc       = matthews_corrcoef(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )
    cm = confusion_matrix(all_labels, all_preds)
    elapsed = time.time() - t0

    # 8) 결과 출력
    print(f"\n Validation Loss     : {avg_loss:.4f}")
    print(f" Validation Accuracy : {avg_acc:.4f}")
    print(f" Matthews CorrCoef   : {mcc:.4f}")
    print(f" Precision           : {precision*100:.2f}%")
    print(f" Recall              : {recall*100:.2f}%")
    print(f" F1-score            : {f1*100:.2f}%")
    print(" Confusion Matrix:\n", cm)
    print(f" Validation Time     : {elapsed:.1f}s\n")

    return {
        "loss": avg_loss,
        "accuracy": avg_acc,
        "mcc": mcc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
    }

# 사용 예시
qnli_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=64, collate_fn=data_collator
)
metrics_qnli = run_validation_qnli(model, qnli_dataloader, device)


## RTE all

### RTE

In [ ]:
import tensorflow as tf

# Get the GPU device name.
device_name = tf.test.gpu_device_name()

# The device name should look like the following:
if device_name == '/device:GPU:0':
    print('Found GPU at: {}'.format(device_name))
else:
    raise SystemError('GPU device not found')

import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
!pip install --upgrade sympy
!pip install --upgrade fsspec datasets
!pip install transformers
!pip install wget
!pip install datasets
!pip install scikit-learn
!pip install keras
!pip install evaluate
import os
import wget
import pandas as pd
import torch
import numpy as np
import time, datetime

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.optim import AdamW

checkpoint = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(checkpoint)
model      = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2   # RTE도 entailment vs non-entailment 이므로 2-class
)

# 1) RTE용 예시: (premise, hypothesis) 쌍
sequences = [
    ("A soccer game with multiple males playing.",
     "Some men are playing a sport."),    # entailment
    ("An older and younger man smiling.",
     "Two men are smiling and laughing at the cats playing on the floor."),  # non-entailment
]
batch = tokenizer(
    [p for p, h in sequences],
    [h for p, h in sequences],
    padding=True, truncation=True, return_tensors="pt"
)

# 2) 이진 레이블: entailment=1, non-entailment=0
batch["labels"] = torch.tensor([1, 0], dtype=torch.long)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,         # GLUE 논문 권장 범위
    weight_decay=0.01
)

# 3) forward + backward
outputs = model(**batch)
loss = outputs.loss
loss.backward()
optimizer.step()


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# 1) QNLI 데이터셋 로드
raw_datasets = load_dataset("glue", "rte")
tokenizer    = AutoTokenizer.from_pretrained("bert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer)

# 2) tokenize 함수: question ↔ sentence 쌍
def tokenize_function(examples):
    return tokenizer(
        examples["sentence1"],
        examples["sentence2"],
        truncation=True,
        max_length=8  #================================================================================================================================================================
    )

# 3) 토크나이징 & 컬럼 정리
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

# 4) 확인
print(tokenized_datasets["train"].column_names)
# → ['input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [ ]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=64, collate_fn=data_collator
)

for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2   # RTE는 entailment vs non-entailment, 2-class 그대로
)
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)  # logits: (batch_size, 2)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,            # GLUE 논문 권장 learning rate (2e-5 ~ 5e-5)
    weight_decay=0.01   # GLUE 논문 권장 weight decay
)

from transformers import get_scheduler

num_epochs = 3                                # GLUE 기본 epoch 수
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",                                  # GLUE 논문 기본 scheduler (linear decay)
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),  # GLUE 논문 권장 warmup 10%
    num_training_steps=num_training_steps,
)
print(num_training_steps)

import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
device


In [ ]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
model.to(device)


In [ ]:
import evaluate
from tqdm.auto import tqdm

metric = evaluate.load("glue", "rte")
model.eval()

with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Evaluating", unit="batch"):
        labels = batch["labels"]
        inputs = {k: v.to(device) for k,v in batch.items() if k!="labels"}

        outputs = model(**inputs)
        logits  = outputs.logits       # (batch, 2)
        preds   = torch.argmax(logits, dim=-1).cpu()

        metric.add_batch(predictions=preds, references=labels)

print(metric.compute())  # {'accuracy': …}


### RTE 구조 적용

In [ ]:
import torch
from transformers.models.bert.modeling_bert import BertSelfAttention

class BertSelfAttentionModified(BertSelfAttention):
    def forward(self,
                hidden_states,
                attention_mask=None,
                head_mask=None,
                encoder_hidden_states=None,
                encoder_attention_mask=None,
                past_key_value=None,
                output_attentions=False):

        # 1) query/key/value (원본과 동일)
        mixed_query_layer = self.query(hidden_states)
        key_layer   = self.transpose_for_scores(
                          self.key(encoder_hidden_states
                                   if encoder_hidden_states is not None
                                   else hidden_states)
                      )
        value_layer = self.transpose_for_scores(
                          self.value(encoder_hidden_states
                                     if encoder_hidden_states is not None
                                     else hidden_states)
                      )
        query_layer = self.transpose_for_scores(mixed_query_layer)

        # 2) 스코어 계산 및 scaling
        attention_scores = torch.matmul(
            query_layer, key_layer.transpose(-1, -2)
        ) / (self.attention_head_size ** 0.5)
        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        # 3) CORDIC-Softmax 교체

        B, H, L, _ = attention_scores.size()           # e.g. 1,12,10,10
        flat = attention_scores.view(-1, L)            # [B*H, L]
        rows = []

       # for row in flat:
       #     # (1) 안정화: max 빼기
       #     #row = row - row.max()
       #     # (2) 정수부 범위 클램핑 (–63.999 ~ +63.999)
       #     row = row.clamp(min=-63.999, max=63.999)
       #     # (3) Python float 리스트로 변환 → top()
       #     out = top(*row.tolist())
       #     # (4) 다시 tensor 생성
       #     rows.append(torch.tensor(
       #         out,
       #         dtype=attention_scores.dtype,
       #         device=attention_scores.device
       #     ))
        for row in flat:
            row = row.clamp(min=-63.999, max=63.999)
            row_list = row.tolist()
            out = top(row_list, L)
            rows.append(torch.tensor(
                out, dtype=attention_scores.dtype, device=attention_scores.device
            ))

        attention_probs = torch.stack(rows).view(B, H, L, L)
        # ─────────────────────────────────────────────

        # 4) dropout & context 계산 (원본과 동일)
        attention_probs = self.dropout(attention_probs)
        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(new_shape)

        outputs = (context_layer, attention_probs) \
                  if output_attentions else (context_layer,)
        return outputs

config = model.config
for layer in model.bert.encoder.layer:
    orig = layer.attention.self
    # Modified 모듈 생성
    mod = BertSelfAttentionModified(config)
    # 기존 가중치 로드
    mod.load_state_dict(orig.state_dict(), strict=False)

    mod.to(device)
    # 교체
    layer.attention.self = mod

In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (
    matthews_corrcoef,
    precision_recall_fscore_support,
    confusion_matrix
)

def run_validation_rte(model, validation_dataloader, device):
    # 1) Self-Attention 모듈 확인
    print("== Self-Attention Modules (QNLI) ==")
    for idx, layer in enumerate(model.bert.encoder.layer):
        cls_name = layer.attention.self.__class__.__name__
        print(f"  Layer {idx:2d}: {cls_name}")
    print("===============================\n")

    # 2) device에 올리고 eval 모드
    model.to(device)
    model.eval()

    criterion     = nn.CrossEntropyLoss()
    eval_loss     = 0.0
    nb_eval_steps = 0

    all_preds  = []
    all_labels = []

    t0 = time.time()
    for batch in tqdm(validation_dataloader, desc="Validating RTE"):
        # 3) batch를 device로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # 4) forward
        with torch.no_grad():
            outputs = model(
                input_ids      = batch["input_ids"],
                token_type_ids = batch["token_type_ids"],
                attention_mask = batch["attention_mask"]
            )
        logits = outputs.logits  # [B, 2]

        # 5) loss 누적
        loss = criterion(logits, batch["labels"])
        eval_loss += loss.item()
        nb_eval_steps += 1

        # 6) preds & labels 수집
        preds  = np.argmax(logits.cpu().numpy(), axis=1)
        labels = batch["labels"].cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    # 7) 평균 지표 계산
    avg_loss  = eval_loss / nb_eval_steps
    avg_acc   = np.mean(np.array(all_preds) == np.array(all_labels))
    mcc       = matthews_corrcoef(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )
    cm = confusion_matrix(all_labels, all_preds)
    elapsed = time.time() - t0

    # 8) 결과 출력
    print(f"\n Validation Loss     : {avg_loss:.4f}")
    print(f" Validation Accuracy : {avg_acc:.4f}")
    print(f" Matthews CorrCoef   : {mcc:.4f}")
    print(f" Precision           : {precision*100:.2f}%")
    print(f" Recall              : {recall*100:.2f}%")
    print(f" F1-score            : {f1*100:.2f}%")
    print(" Confusion Matrix:\n", cm)
    print(f" Validation Time     : {elapsed:.1f}s\n")

    return {
        "loss": avg_loss,
        "accuracy": avg_acc,
        "mcc": mcc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
    }

# 사용 예시
rte_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=64, collate_fn=data_collator
)
metrics_rte = run_validation_rte(model, rte_dataloader, device)


## valiation

In [ ]:
import time
import torch
import torch.nn as nn
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import (
    matthews_corrcoef,
    precision_recall_fscore_support,
    confusion_matrix
)

def run_validation_mrpc(model, validation_dataloader, device):
    # 1) Self-Attention 모듈 확인
    print("== Self-Attention Modules ==")
    for idx, layer in enumerate(model.bert.encoder.layer):
        cls_name = layer.attention.self.__class__.__name__
        print(f"  Layer {idx:2d}: {cls_name}")
    print("=============================\n")

    # 2) 모델을 device에 올리고 eval 모드
    model.to(device)
    model.eval()

    criterion     = nn.CrossEntropyLoss()
    eval_loss     = 0.0
    nb_eval_steps = 0

    all_preds  = []
    all_labels = []

    t0 = time.time()
    for batch in tqdm(validation_dataloader, desc="Validating MRPC"):
        # 배치 내 모든 텐서를 device로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        b_input_ids      = batch["input_ids"]
        b_token_type_ids = batch["token_type_ids"]
        b_attention_mask = batch["attention_mask"]
        b_labels         = batch["labels"]

        # forward
        with torch.no_grad():
            outputs = model(
                input_ids      = b_input_ids,
                token_type_ids = b_token_type_ids,
                attention_mask = b_attention_mask
            )
        logits = outputs.logits  # [B, 2]

        # loss 누적
        loss = criterion(logits, b_labels)
        eval_loss += loss.item()
        nb_eval_steps += 1

        # 예측값 & 정답 수집
        preds  = np.argmax(logits.cpu().numpy(), axis=1)
        labels = b_labels.cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    # 평균 지표 계산
    avg_loss  = eval_loss / nb_eval_steps
    avg_acc   = np.mean(np.array(all_preds) == np.array(all_labels))
    mcc       = matthews_corrcoef(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary'
    )
    cm = confusion_matrix(all_labels, all_preds)
    elapsed = time.time() - t0

    # 결과 출력
    print(f"\nValidation Loss     : {avg_loss:.4f}")
    print(f"Validation Accuracy : {avg_acc:.4f}")
    print(f"Matthews CorrCoef   : {mcc:.4f}")
    print(f"Precision           : {precision*100:.2f}%")
    print(f"Recall              : {recall*100:.2f}%")
    print(f"F1-score            : {f1*100:.2f}%")
    print("Confusion Matrix:\n", cm)
    print(f"Validation Time     : {elapsed:.1f}s\n")

    return {
        "loss": avg_loss,
        "accuracy": avg_acc,
        "mcc": mcc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
    }

# 사용 예시
metrics = run_validation_mrpc(model, eval_dataloader, device)
